In [ ]:
from typing import Any
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
import kss
import re

def document_ocr(document_path : str, file_name : str) -> Any:
    """(로컬용) 문서 경로를 받아서 OCR을 수행하는 함수 / 우선은 PDF만 지원"""
    if not document_path.endswith(".pdf"):
        return "Error: Unsupported file type"

    refined_full_text = ""
    tesseract_config = "--psm 3 -l kor+eng"
    try:
        with pdfplumber.open(document_path) as f:
            total_pages = len(f.pages)
            for page_num, page in enumerate(f.pages, 1):
                # 디지털 구분 - chars가 전체 텍스트 길이의 70% 이상이면 디지털로 처리
                extracted_text = page.extract_text()
                is_digital = False
                if extracted_text and extracted_text.strip():
                    if len(page.chars) > len(extracted_text.strip()) * 0.7:
                        is_digital = True
                # 디지털 처리
                if is_digital:
                    current_text = extracted_text
                    source_type = "Digital"
                else:
                    # 디지털 처리 실패 시 OCR 진행
                    print(f"Page {page_num}/{total_pages} performing OCR...")
                    images = convert_from_path(document_path, first_page = page_num, last_page = page_num)
                    if images:
                        current_text = pytesseract.image_to_string(images[0], config = tesseract_config)
                        source_type = "OCR"
                    else:
                        continue
                
                # 문장 구분
                text = re.sub(r'\n{2,}', '[[PARAGRAPH]]', current_text)
                lines = text.split('\n')
                processed_text = ""
                for i in range(len(lines)):
                    line = lines[i].rstrip()
                    if not line:
                        continue
                    if i < len(lines) - 1:
                        next_line = lines[i + 1].strip()
                        is_sentence_end = re.search(r'[.?!함다요임)\]>"\']$', line)
                        is_next_start_marker = re.match(r'^[※*○●□■→\->\=>\d+\.\[]', next_line)
                        if is_sentence_end or is_next_start_marker:
                            processed_text += line + "\n"
                        else:
                            if re.search(r'[가-힣]$', line) and re.match(r'^[가-힣]', next_line):
                                processed_text += line
                            else:
                                processed_text += line + " "
                    else:
                        processed_text += line
                page_processed_text = processed_text.replace("[[PARAGRAPH]]", "\n\n")
                try:
                    page_processed_text = "\n".join(kss.split_sentences(page_processed_text))
                except:
                    pass

                refined_full_text += f"--- [Page {page_num} ({source_type})] ---\n{page_processed_text}\n"
                print(f"Page {page_num}/{total_pages} processed")

        # OCR 결과 저장 (확인용..)
        with open(f"data/test/{file_name}.txt", "w", encoding = "utf-8") as f:
            f.write(refined_full_text)
            print(f"OCR 결과가 data/test/{file_name}.txt에 저장되었습니다.")

    except Exception as e:
        return f"Error: {e}"
    
    return refined_full_text

In [5]:
document_path = "data/test/test_근로계약서.pdf"
file_name = "test_근로계약서"
ocr_result = document_ocr(document_path, file_name)
print(ocr_result)

len(page.chars) : 505
len(extracted_text.strip()) : 520


c:\Users\SAMSUNG\Desktop\Grad_School\RAG_LAW\.venv\Lib\site-packages\pecab\_tokenizer.py:265: RuntimeWarning: overflow encountered in scalar add
  from_pos_data.costs[idx]
c:\Users\SAMSUNG\Desktop\Grad_School\RAG_LAW\.venv\Lib\site-packages\pecab\_tokenizer.py:274: RuntimeWarning: overflow encountered in scalar add
  least_cost += word_cost


Page 1/6 processed
len(page.chars) : 719
len(extracted_text.strip()) : 738
Page 2/6 processed
len(page.chars) : 795
len(extracted_text.strip()) : 813
Page 3/6 processed
len(page.chars) : 646
len(extracted_text.strip()) : 660
Page 4/6 processed
len(page.chars) : 728
len(extracted_text.strip()) : 743
Page 5/6 processed
len(page.chars) : 90
len(extracted_text.strip()) : 95
Page 6/6 processed
OCR 결과가 data/test/test_근로계약서.txt에 저장되었습니다.
--- [Page 1 (Digital)] ---
Ⅰ. 근로계약서 샘플 (1/2)
근로계약서 AAA회사(이하 회사)와BB B(이하직원 )는다음과같이근로계약을체결한다.
제1 조【업무내용및근무 장소】 직원 은 2021년 ○월○ 일(근로관계개시 일)부터회사의지시에따라다음과같이근무하 며직원 의업무및근무 장소 는
회사의 업무상 필요 에 따라 추후변경될수있 다.
1 .업무 :2 .근무장소: 가.
서울
나.
재택 근
무시 :자택등제 2 조【 수습기간 】직원의수습기간은근로관계개시 일로부터3개월 로한다 .
단,회사 는필요하 거나적절하다고생각하는 경우위 수습 기간을 생략,단축 또는연장할 수있다.
수습 기간중또는수습 기간만료시에 계속 근로가 부적당하다고인정하는경우사전 예고및기타보상없이 본계약을해지 할수있으며, 그경우회 사는 직원의실제근무일수에대해서만 급여를지급 한다.
제 3조【업무수행】직원은본계약에명시 된사항및회사의규정 및지시사항을성실 히준수및이행하여야 한다.
페이지 2/2
--- [Page 2 (Digital)] ---
제4조 【급여】 ① 월 급여는 ○,000,000원으로 한다.
1. 월 급여에는 식대 100,000

#### pytesseract

In [5]:
def pytessea_ocr(document_path : str) -> Any:
    if document_path.endswith(".pdf"):
        with pdfplumber.open(document_path) as f:
            total_pages = len(f.pages)
            for page_num, page in enumerate(f.pages, 1):
                images = convert_from_path(document_path, first_page = page_num, last_page = page_num)
                if images:
                    ocr_result = pytesseract.image_to_string(images[0], lang = "kor+eng")
                    print(ocr_result)
                    full_text += f"\n--- [Page {page_num} (OCR)] ---\n{ocr_result}\n"
                else:
                    print(f"Page {page_num} of {total_pages} skipped (no images)")
        return full_text

In [7]:
document_path = "data/test/test_pdf.pdf"
ocr_result = pytessea_ocr(document_path)

18019에서 협약한 과제의 변경 건은 '본교 연구포탈'과 IRIS 시스템'에서 모두 변경신청을 통해 반영해주셔

야 합니다.

매뉴얼을 참고하시어 신규 연구원 참여 SSS IRIS 제출해주시고 제게 기관 승인요청해주시기 바랍니

IRIS 시스템에서 협약한 연구 과제는 1815에서 협약변경을 진행해야 합니다.

※ 1819 연구원변경 통보 입력방법

URIS - 8&0업무포털 - 과제수행 - (승인통보)협약변경신청|에서 담당 과제를 조회하신 뒤,
<변경신청> 탭에서 변경내용 및 사유 입력 후 저상 -> <연구기관> 탭에서 "변경항목 선택"으로 연구원변
경(통보) 선택

> 1단계 1차년도 고려대학교 연구
입, 연구자전환동의* 및 연구윤리동
> <최종학인> 탭에서 최종확인 후 제출 -> 산단 담당자에게 검토 및 승인 요청(메일로 요청)

+ 연구자전환동의 : 해당 연구원이 RIS 로그인 후, 메인화면 우측의 Quick Link '국가연구자정보시스
템'에 들어가서 팝업에 안내되는대로 동의 진행
++ 연구윤리동의 : 과제별로 참여연구원들은 반드시 동의해야함. '연구윤리동의안내' 처리
동의 절차 진행

> 연구자전환동의 및 연구윤리동의는 정확한 매뉴얼을 알지 못하여, 문의사항은 IRIS 콜센터(1877-2041)
에 연락바랍니다.

연구원별로

ov

※ 189 예산변경 통보 입력방법

URIS - 86&0업무포털 - 과제수행 - (승인통보)협약변경신청]에서 담당 과제를 조회하신 뒤,
<변경신청> 탭에서 변경내용 및 사유 입력 후 저장 -> <연구개발비> 탭에서 "변경항목 선택"으로 '통보-그
외세목변경' 선택

> <비목별 연구비 구성>에서 변경하고자 하는 예산으로 수성 후 저장 — <최종확인> 탭에서 최종확인 후
제출 - 산단 담당자에게 검토 및 승인 요청(메일로 요청)

관련 매뉴얼을 첨부드러오니 참고해주시기 바라며, 시스템 변경신청하는 절차가 복잡하여 어려우실 수 있

산 및 참여연구원 변경 신청 건마다 RISA 신청해주셔야 연구비 집행 및 정산 시 문제없으므

UnboundLocalError: cannot access local variable 'full_text' where it is not associated with a value